In [2]:
from pathlib import Path
import pandas as pd

# Chemin vers les données nettoyées
DOSSIER = Path("../data/processed/date=2026-10-29")

# Chargement des données
lignes = pd.read_csv(DOSSIER / "lignes_commande.csv")
commandes = pd.read_csv(DOSSIER / "commandes.csv")
clients = pd.read_csv(DOSSIER / "clients.csv")

# Vérification du chargement
print("Lignes de commande :", len(lignes))
print("Commandes :", len(commandes))
print("Clients :", len(clients))

Lignes de commande : 9608
Commandes : 3920
Clients : 3333


In [3]:
ventes = lignes.merge(
    commandes,
    on="commande_id",
    how="left"
)

print("Nombre de lignes de vente :", len(ventes))
print(ventes.head())

Nombre de lignes de vente : 9608
  ligne_commande_id commande_id  produit_id  quantite  prix_unitaire_xof  \
0        LIG0000001  CMD0000001  PRD0000019         1               2000   
1        LIG0000002  CMD0000001  PRD0000010         1              35000   
2        LIG0000003  CMD0000002  PRD0000018         2               2600   
3        LIG0000004  CMD0000002  PRD0000005         1               9500   
4        LIG0000005  CMD0000003  PRD0000010         4              35000   

   montant_xof   client_id date_commande  statut  
0         2000  CLI0000343    2025-10-30  livree  
1        35000  CLI0000343    2025-10-30  livree  
2         5200  CLI0000448    2025-10-30  livree  
3         9500  CLI0000448    2025-10-30  livree  
4       140000  CLI0000549    2025-10-30  livree  


In [4]:
ventes = ventes.merge(
    clients[["client_id", "ville", "region"]],
    on="client_id",
    how="left"
)

print(ventes[["commande_id", "client_id", "ville", "region", "montant_xof"]].head())

  commande_id   client_id   ville region  montant_xof
0  CMD0000001  CLI0000343   Dakar  Dakar         2000
1  CMD0000001  CLI0000343   Dakar  Dakar        35000
2  CMD0000002  CLI0000448  Pikine  Dakar         5200
3  CMD0000002  CLI0000448  Pikine  Dakar         9500
4  CMD0000003  CLI0000549   Thiès  Thiès       140000


In [5]:
# Regrouper les ventes par ville
ca_par_ville = (
    ventes
    .groupby("ville", dropna=False)["montant_xof"]
    .sum()
    .sort_values(ascending=False)
)

print(ca_par_ville)

ville
Dakar           245729443
Mbour            56474392
Inconnue         52722070
Pikine           46204769
Saint-Louis      45333358
Touba            39797528
Thiès            38894044
Kaolack          37143297
Guédiawaye       34970692
Ziguinchor       34827568
Rufisque         27912132
Louga            16953295
Diourbel         10834860
Fatick            9817269
Kaffrine          8916490
Tivaouane         8328505
Tambacounda       7078247
Richard-Toll      4609209
Matam             4193001
Kolda             3721518
Sédhiou           3400175
Kédougou          1890363
Name: montant_xof, dtype: int64


In [6]:
ca_par_region = (
    ventes
    .groupby("region", dropna=False)["montant_xof"]
    .sum()
    .sort_values(ascending=False)
)

print(ca_par_region)

region
Dakar          354817036
Thiès          103696941
Inconnue        52722070
Diourbel        50632388
Saint-Louis     49942567
Kaolack         37143297
Ziguinchor      34827568
Louga           16953295
Fatick           9817269
Kaffrine         8916490
Tambacounda      7078247
Matam            4193001
Kolda            3721518
Sédhiou          3400175
Kédougou         1890363
Name: montant_xof, dtype: int64


In [7]:
ca_total = ventes["montant_xof"].sum()
ca_villes_total = ca_par_ville.sum()
ca_regions_total = ca_par_region.sum()

print(f"Chiffre d'affaires total : {ca_total:,.0f} XOF")
print(f"Somme du CA par ville   : {ca_villes_total:,.0f} XOF")
print(f"Somme du CA par région  : {ca_regions_total:,.0f} XOF")

print("\nVérification par ville :", ca_total == ca_villes_total)
print("Vérification par région :", ca_total == ca_regions_total)

Chiffre d'affaires total : 739,752,225 XOF
Somme du CA par ville   : 739,752,225 XOF
Somme du CA par région  : 739,752,225 XOF

Vérification par ville : True
Vérification par région : True


In [8]:
# Remplacer les villes manquantes ou vides
ventes["ville"] = ventes["ville"].fillna("Inconnue").replace(
    r"^\s*$", "Inconnue", regex=True
)

# Remplacer les régions manquantes ou vides
ventes["region"] = ventes["region"].fillna("Inconnue").replace(
    r"^\s*$", "Inconnue", regex=True
)

# Recalculer le chiffre d'affaires par ville
ca_par_ville = (
    ventes.groupby("ville")["montant_xof"]
    .sum()
    .sort_values(ascending=False)
)

# Recalculer le chiffre d'affaires par région
ca_par_region = (
    ventes.groupby("region")["montant_xof"]
    .sum()
    .sort_values(ascending=False)
)

print("CHIFFRE D'AFFAIRES PAR RÉGION")
print(ca_par_region)

CHIFFRE D'AFFAIRES PAR RÉGION
region
Dakar          354817036
Thiès          103696941
Inconnue        52722070
Diourbel        50632388
Saint-Louis     49942567
Kaolack         37143297
Ziguinchor      34827568
Louga           16953295
Fatick           9817269
Kaffrine         8916490
Tambacounda      7078247
Matam            4193001
Kolda            3721518
Sédhiou          3400175
Kédougou         1890363
Name: montant_xof, dtype: int64


In [9]:
# Calcul du CA total directement depuis les lignes de commande
ca_total = lignes["montant_xof"].sum()

# Somme des CA calculés par ville et par région
total_par_ville = ca_par_ville.sum()
total_par_region = ca_par_region.sum()

print(f"CA total des lignes de commande : {ca_total:,.0f} XOF")
print(f"Somme du CA par ville           : {total_par_ville:,.0f} XOF")
print(f"Somme du CA par région          : {total_par_region:,.0f} XOF")

print("\nVérification par ville :", ca_total == total_par_ville)
print("Vérification par région :", ca_total == total_par_region)

CA total des lignes de commande : 739,752,225 XOF
Somme du CA par ville           : 739,752,225 XOF
Somme du CA par région          : 739,752,225 XOF

Vérification par ville : True
Vérification par région : True


In [10]:
print("Lignes de commande au départ :", len(lignes))
print("Lignes après les deux jointures :", len(ventes))
print("Identifiants de commande distincts :", commandes["commande_id"].nunique())
print("Identifiants de client distincts :", clients["client_id"].nunique())

Lignes de commande au départ : 9608
Lignes après les deux jointures : 9608
Identifiants de commande distincts : 3920
Identifiants de client distincts : 3333


In [11]:
produits = pd.read_csv(DOSSIER / "produits.csv")

In [12]:
ventes_produits = lignes.merge(
    produits[["produit_id", "nom_produit", "categorie"]],
    on="produit_id",
    how="left"
)

print(ventes_produits.head())

  ligne_commande_id commande_id  produit_id  quantite  prix_unitaire_xof  \
0        LIG0000001  CMD0000001  PRD0000019         1               2000   
1        LIG0000002  CMD0000001  PRD0000010         1              35000   
2        LIG0000003  CMD0000002  PRD0000018         2               2600   
3        LIG0000004  CMD0000002  PRD0000005         1               9500   
4        LIG0000005  CMD0000003  PRD0000010         4              35000   

   montant_xof           nom_produit     categorie  
0         2000      Café Touba 500 g  alimentation  
1        35000       Bazin riche 5 m   mode_tissus  
2         5200  Lait en poudre 400 g  alimentation  
3         9500    Écouteurs sans fil    telephonie  
4       140000       Bazin riche 5 m   mode_tissus  


In [13]:
top10_quantite = (
    ventes_produits
    .groupby(["produit_id", "nom_produit"], as_index=False)
    .agg(quantite_totale=("quantite", "sum"))
    .sort_values("quantite_totale", ascending=False)
    .head(10)
)

display(top10_quantite)

,produit_id,nom_produit,quantite_totale
17,PRD0000018,Lait en poudre 400 g,3759
29,PRD0000030,Mixeur blender,2065
9,PRD0000010,Bazin riche 5 m,1356
2,PRD0000003,Smartphone Itel A70,1062
11,PRD0000012,Robe en wax,899
28,PRD0000029,Téléviseur LED 43 pouces,710
23,PRD0000024,Set de casseroles,581
27,PRD0000028,Climatiseur 12 000 BTU,557
20,PRD0000021,Oignons 25 kg,463
4,PRD0000005,Écouteurs sans fil,429


In [14]:
top10_quantite = (
    ventes_produits
    .groupby(["produit_id", "nom_produit"], as_index=False)
    .agg(quantite_totale=("quantite", "sum"))
    .sort_values("quantite_totale", ascending=False)
    .head(10)
)

display(top10_quantite)

,produit_id,nom_produit,quantite_totale
17,PRD0000018,Lait en poudre 400 g,3759
29,PRD0000030,Mixeur blender,2065
9,PRD0000010,Bazin riche 5 m,1356
2,PRD0000003,Smartphone Itel A70,1062
11,PRD0000012,Robe en wax,899
28,PRD0000029,Téléviseur LED 43 pouces,710
23,PRD0000024,Set de casseroles,581
27,PRD0000028,Climatiseur 12 000 BTU,557
20,PRD0000021,Oignons 25 kg,463
4,PRD0000005,Écouteurs sans fil,429


In [15]:
print("=== Vérification des données ===")
print("Lignes de commande :", len(lignes))
print("Commandes :", len(commandes))
print("Clients :", len(clients))

print("\n=== Vérification des variables ===")
variables = [
    "ventes",
    "ventes_produits",
    "top10_quantite",
    "top10_ca",
    "montants_commandes",
    "navigation",
    "cohortes",
    "retention",
    "paiements",
]

for nom in variables:
    print(f"{nom} :", "disponible" if nom in globals() else "à calculer")

=== Vérification des données ===
Lignes de commande : 9608
Commandes : 3920
Clients : 3333

=== Vérification des variables ===
ventes : disponible
ventes_produits : disponible
top10_quantite : disponible
top10_ca : à calculer
montants_commandes : à calculer
navigation : à calculer
cohortes : à calculer
retention : à calculer
paiements : à calculer


In [17]:
top10_ca = (
    ventes_produits
    .groupby(["produit_id", "nom_produit"], as_index=False)
    .agg(ca_total=("montant_xof", "sum"))
    .sort_values("ca_total", ascending=False)
    .head(10)
)

top10_ca

,produit_id,nom_produit,ca_total
27,PRD0000028,Climatiseur 12 000 BTU,159012035
28,PRD0000029,Téléviseur LED 43 pouces,124246518
36,PRD0000037,Ordinateur portable 15 pouces,58831205
2,PRD0000003,Smartphone Itel A70,47674600
9,PRD0000010,Bazin riche 5 m,47440096
29,PRD0000030,Mixeur blender,38191091
38,PRD0000039,Imprimante multifonction,34343416
31,PRD0000032,Cuisinière 4 feux,26970491
3,PRD0000004,iPhone 13 reconditionné,24607761
26,PRD0000027,Réfrigérateur 180 L,23500686


In [18]:
# Total des ventes pour chaque commande
montants_commandes = (
    lignes
    .groupby("commande_id", as_index=False)
    .agg(montant_commande=("montant_xof", "sum"))
)

# Panier moyen par commande
panier_moyen = montants_commandes["montant_commande"].mean()

# Montant moyen d'une ligne de commande
moyenne_par_ligne = lignes["montant_xof"].mean()

print(f"Nombre de commandes avec des lignes : {len(montants_commandes)}")
print(f"Panier moyen : {panier_moyen:,.0f} XOF")
print(f"Montant moyen par ligne : {moyenne_par_ligne:,.0f} XOF")

Nombre de commandes avec des lignes : 3904
Panier moyen : 189,486 XOF
Montant moyen par ligne : 76,993 XOF


In [19]:
navigation = pd.read_csv(DOSSIER / "navigation.csv")

# Nombre total de sessions distinctes
sessions_total = navigation["session_id"].nunique()

# Sessions ayant atteint l'étape du paiement
sessions_paiement = navigation.loc[
    navigation["type_evenement"] == "debut_paiement",
    "session_id"
].nunique()

taux_conversion = (
    sessions_paiement / sessions_total * 100
    if sessions_total > 0 else 0
)

print(f"Sessions totales : {sessions_total}")
print(f"Sessions avec début de paiement : {sessions_paiement}")
print(f"Taux de conversion : {taux_conversion:.2f} %")

Sessions totales : 8000
Sessions avec début de paiement : 1253
Taux de conversion : 15.66 %


In [20]:
# 1. Préparer les dates des commandes
commandes["date_commande"] = pd.to_datetime(
    commandes["date_commande"],
    errors="coerce"
)

commandes_valides = commandes.dropna(
    subset=["client_id", "date_commande"]
).copy()

# 2. Identifier le mois de chaque commande
commandes_valides["mois_commande"] = (
    commandes_valides["date_commande"].dt.to_period("M")
)

# 3. Trouver la première commande de chaque client
premiere_commande = (
    commandes_valides.groupby("client_id")["date_commande"]
    .min()
    .rename("premiere_commande")
)

commandes_valides = commandes_valides.join(
    premiere_commande,
    on="client_id"
)

# 4. Définir la cohorte de chaque client
commandes_valides["cohorte"] = (
    commandes_valides["premiere_commande"].dt.to_period("M")
)

# 5. Calculer le nombre de mois depuis la première commande
commandes_valides["mois_cohorte"] = (
    (commandes_valides["mois_commande"].dt.year
     - commandes_valides["cohorte"].dt.year) * 12
    + commandes_valides["mois_commande"].dt.month
    - commandes_valides["cohorte"].dt.month
)

# 6. Construire la matrice des cohortes
cohortes = (
    commandes_valides
    .groupby(["cohorte", "mois_cohorte"])["client_id"]
    .nunique()
    .unstack(fill_value=0)
)

# 7. Calculer le taux de rétention
retention = cohortes.div(cohortes[0], axis=0) * 100

print("=== Nombre de clients par cohorte ===")
display(cohortes)

print("=== Taux de rétention (%) ===")
display(retention.round(1))

=== Nombre de clients par cohorte ===


mois_cohorte,0,1,2,3,4,5,6,7,8,9,10,11,12
cohorte,,,,,,,,,,,,,
2025-10,12,3,5,2,3,2,4,3,5,2,4,8,3
2025-11,176,37,34,40,42,48,36,39,34,46,41,38,0
2025-12,187,42,24,31,36,34,31,34,27,37,24,0,0
2026-01,136,20,17,25,23,20,23,29,17,20,0,0,0
2026-02,108,10,14,13,10,15,13,16,6,0,0,0,0
2026-03,104,14,9,10,15,11,19,14,0,0,0,0,0
2026-04,118,17,14,16,16,17,20,0,0,0,0,0,0
2026-05,118,21,11,21,14,13,0,0,0,0,0,0,0
2026-06,135,20,19,21,22,0,0,0,0,0,0,0,0


=== Taux de rétention (%) ===


mois_cohorte,0,1,2,3,4,5,6,7,8,9,10,11,12
cohorte,,,,,,,,,,,,,
2025-10,100.0,25.0,41.7,16.7,25.0,16.7,33.3,25.0,41.7,16.7,33.3,66.7,25.0
2025-11,100.0,21.0,19.3,22.7,23.9,27.3,20.5,22.2,19.3,26.1,23.3,21.6,0.0
2025-12,100.0,22.5,12.8,16.6,19.3,18.2,16.6,18.2,14.4,19.8,12.8,0.0,0.0
2026-01,100.0,14.7,12.5,18.4,16.9,14.7,16.9,21.3,12.5,14.7,0.0,0.0,0.0
2026-02,100.0,9.3,13.0,12.0,9.3,13.9,12.0,14.8,5.6,0.0,0.0,0.0,0.0
2026-03,100.0,13.5,8.7,9.6,14.4,10.6,18.3,13.5,0.0,0.0,0.0,0.0,0.0
2026-04,100.0,14.4,11.9,13.6,13.6,14.4,16.9,0.0,0.0,0.0,0.0,0.0,0.0
2026-05,100.0,17.8,9.3,17.8,11.9,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026-06,100.0,14.8,14.1,15.6,16.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
paiements = pd.read_csv(DOSSIER / "paiements.csv")

# CA calculé à partir des lignes de commande
ca_commandes = lignes["montant_xof"].sum()

# Identifier les paiements réussis
paiements_reussis = paiements[
    paiements["statut_paiement"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("reussi")
].copy()

# Total des montants des paiements réussis
ca_paye = paiements_reussis["montant_xof"].sum()

# Commandes ayant au moins un paiement réussi
commandes_payees = paiements_reussis["commande_id"].dropna().unique()

# Commandes sans paiement réussi
commandes_sans_paiement = commandes[
    ~commandes["commande_id"].isin(commandes_payees)
]

print(f"CA des lignes de commande : {ca_commandes:,.0f} XOF")
print(f"Montant des paiements réussis : {ca_paye:,.0f} XOF")
print(f"Écart entre les deux montants : {ca_commandes - ca_paye:,.0f} XOF")
print(f"Commandes sans paiement réussi : {len(commandes_sans_paiement)}")

CA des lignes de commande : 739,752,225 XOF
Montant des paiements réussis : 669,168,696 XOF
Écart entre les deux montants : 70,583,529 XOF
Commandes sans paiement réussi : 486
